# Credit Risk — Feature Engineering

__Objective__

Develop behavioral and financial features based on
the findings from exploratory data analysis.

The engineered features aim to capture:

- Delinquency severity, frequency and recency.
- Historical billing behavior.
- Payment patterns and trends.
- Credit utilization.
- Payment-to-bill relationships.

Original predictors will be preserved to enable
comparisons between baseline and engineered
feature sets.

# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)

# 2. Data Loading

In [3]:
# Load the original dataset
dataset = fetch_ucirepo(id=350)

X = dataset.data.features.copy()
y = dataset.data.targets.copy().squeeze()

# Rename variables
column_mapping = {
    "X1": "LIMIT_BAL",
    "X2": "SEX",
    "X3": "EDUCATION",
    "X4": "MARRIAGE",
    "X5": "AGE",
    **{f"X{i+5}": f"PAY_{i}" for i in range(1, 7)},
    **{f"X{i+11}": f"BILL_AMT{i}" for i in range(1, 7)},
    **{f"X{i+17}": f"PAY_AMT{i}" for i in range(1, 7)}
}

X = X.rename(columns=column_mapping)

display(X.head())

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0
1,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000
2,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000
3,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000
4,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679


# 3. Train-Test Split

A stratified split is used to preserve the original
target distribution in both subsets.

The test set will be reserved for final evaluation.
Data-dependent preprocessing, feature selection
and model optimization will use training data only.

Since target-based EDA was previously conducted
on the full dataset, the test set cannot be
considered a completely untouched holdout.

In [4]:
# Perform stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Validate dataset sizes and target distribution
split_summary = pd.DataFrame({
    "number_of_clients": [
        len(y),
        len(y_train),
        len(y_test)
    ],
    "default_rate": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["Original", "Train", "Test"])

display(split_summary.round(2))

,number_of_clients,default_rate
Original,30000,22.12
Train,24000,22.12
Test,6000,22.12
